# 05. Model Comparison

개인화 상품 추천 엔진 프로젝트의 Day 4 산출물(비교 파트)입니다.  
이 노트북에서는 Day 2~4에서 확보한 주요 모델의 **정확도 / 다양성 / 커버리지 / 시나리오별 적합성**을 한 장의 비교 관점으로 정리합니다.


## 체크리스트
- [x] 전체 모델 비교표 로드
- [x] 정확도 vs 다양성 vs 커버리지 대시보드 작성
- [x] 일반 유저 / sparse-profile user 시나리오별 해석
- [x] 최종 추천 전략 요약


In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = next(
    (path for path in [Path.cwd(), Path.cwd().parent] if (path / "src").exists()),
    Path.cwd(),
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import FIGURES_DIR, METRICS_DIR

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)


## 1. Day 4 결과 로드


In [2]:
comparison_df = pd.read_csv(METRICS_DIR / "day4_model_comparison.csv")
segment_df = pd.read_csv(METRICS_DIR / "day4_segment_comparison.csv")
weighted_grid_df = pd.read_csv(METRICS_DIR / "day4_hybrid_grid_search.csv")
switching_grid_df = pd.read_csv(METRICS_DIR / "day4_switching_grid_search.csv")
with (METRICS_DIR / "day4_hybrid_summary.json").open(encoding="utf-8") as fp:
    hybrid_summary = json.load(fp)

comparison_df


,users_evaluated,precision@10,recall@10,ndcg@10,map@10,intra_list_diversity,coverage,model_name,cold_start_item_share
0,926.0,0.249676,0.279275,0.357057,0.230615,0.594489,0.180993,weighted_hybrid_alpha_0.85,0.000000
1,926.0,0.247948,0.278027,0.353363,0.226528,0.620134,0.171913,cf_user_pearson_k40,0.000000
2,926.0,0.247192,0.276236,0.351804,0.225715,0.616017,0.211259,switching_hybrid_lt_15,0.001404
3,926.0,0.134665,0.140311,0.175515,0.090632,0.632658,0.029056,popularity_baseline,0.000000
4,926.0,0.016631,0.020695,0.023895,0.010534,0.094736,0.366223,content_tfidf_metadata,0.230022


## 2. 주요 지표 대시보드


In [3]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
plot_columns = [
    ("precision@10", "Precision@10"),
    ("ndcg@10", "NDCG@10"),
    ("coverage", "Coverage"),
    ("intra_list_diversity", "Intra-list Diversity"),
]

for ax, (column, title) in zip(axes.flat, plot_columns, strict=True):
    ordered = comparison_df.sort_values(column, ascending=False)
    sns.barplot(
        data=ordered,
        x=column,
        y="model_name",
        hue="model_name",
        dodge=False,
        legend=False,
        palette="Purples_r",
        ax=ax,
    )
    ax.set_title(title)
    ax.set_ylabel("")

fig.tight_layout()
figure_path = FIGURES_DIR / "day4_model_comparison_dashboard.png"
fig.savefig(figure_path, dpi=200, bbox_inches="tight")
plt.close(fig)

figure_path.relative_to(PROJECT_ROOT)


WindowsPath('artifacts/figures/day4_model_comparison_dashboard.png')

## 3. Accuracy vs Coverage trade-off
Weighted Hybrid가 accuracy와 coverage를 함께 올렸는지, Switching Hybrid가 coverage fallback 역할을 하는지 확인합니다.


In [4]:
tradeoff_df = comparison_df[["model_name", "precision@10", "coverage", "intra_list_diversity"]].copy()

fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    data=tradeoff_df,
    x="coverage",
    y="precision@10",
    size="intra_list_diversity",
    sizes=(100, 700),
    hue="model_name",
    ax=ax,
)
for _, row in tradeoff_df.iterrows():
    ax.text(
        row["coverage"] + 0.002,
        row["precision@10"] + 0.0005,
        row["model_name"],
        fontsize=9,
    )
ax.set_title("Accuracy vs Coverage")
ax.legend(loc="lower right", fontsize=8)

tradeoff_path = FIGURES_DIR / "day4_model_tradeoff.png"
fig.tight_layout()
fig.savefig(tradeoff_path, dpi=200, bbox_inches="tight")
plt.close(fig)

tradeoff_path.relative_to(PROJECT_ROOT)


WindowsPath('artifacts/figures/day4_model_tradeoff.png')

## 4. 시나리오별 best model 정리
- overall accuracy: Precision@10 우선
- exploration / catalog reach: Coverage 우선
- warm user fallback: warm_user segment에서 Precision@10 우선
- power user default: power_user segment에서 Precision@10 우선


In [5]:
overall_best = comparison_df.sort_values(["precision@10", "ndcg@10"], ascending=[False, False]).iloc[0]
coverage_best = comparison_df.sort_values(["coverage", "precision@10"], ascending=[False, False]).iloc[0]
diversity_best = comparison_df.sort_values(["intra_list_diversity", "precision@10"], ascending=[False, False]).iloc[0]

warm_segment = segment_df[segment_df["segment"] == "warm_user"].sort_values(
    ["precision@10", "coverage"], ascending=[False, False]
)
power_segment = segment_df[segment_df["segment"] == "power_user"].sort_values(
    ["precision@10", "coverage"], ascending=[False, False]
)

scenario_df = pd.DataFrame(
    [
        {
            "scenario": "overall_default_ranker",
            "recommended_model": overall_best["model_name"],
            "reason": "Precision@10 / NDCG@10이 가장 높음",
        },
        {
            "scenario": "catalog_exploration",
            "recommended_model": coverage_best["model_name"],
            "reason": "Coverage가 가장 높아 cold item 노출 여지가 큼",
        },
        {
            "scenario": "highest_diversity",
            "recommended_model": diversity_best["model_name"],
            "reason": "Intra-list Diversity가 가장 높음",
        },
        {
            "scenario": "warm_user_segment",
            "recommended_model": warm_segment.iloc[0]["model_name"],
            "reason": "warm_user 구간 Precision@10 기준 최상위",
        },
        {
            "scenario": "power_user_segment",
            "recommended_model": power_segment.iloc[0]["model_name"],
            "reason": "power_user 구간 Precision@10 기준 최상위",
        },
    ]
)

scenario_path = METRICS_DIR / "day4_scenario_best_models.csv"
scenario_df.to_csv(scenario_path, index=False)
scenario_df


,scenario,recommended_model,reason
0,overall_default_ranker,weighted_hybrid_alpha_0.85,Precision@10 / NDCG@10이 가장 높음
1,catalog_exploration,content_tfidf_metadata,Coverage가 가장 높아 cold item 노출 여지가 큼
2,highest_diversity,popularity_baseline,Intra-list Diversity가 가장 높음
3,warm_user_segment,cf_user_pearson_k40,warm_user 구간 Precision@10 기준 최상위
4,power_user_segment,weighted_hybrid_alpha_0.85,power_user 구간 Precision@10 기준 최상위


## 5. 핵심 해석


In [6]:
key_findings = pd.DataFrame(
    [
        {
            "topic": "default ranker",
            "finding": f"{overall_best['model_name']}가 전체 Precision@10, NDCG@10 기준 최고 성능을 보였습니다.",
        },
        {
            "topic": "weighted hybrid",
            "finding": f"best alpha는 {hybrid_summary['best_weighted_alpha']:.2f}이며, CF 단독 대비 accuracy와 coverage를 함께 개선했습니다.",
        },
        {
            "topic": "switching hybrid",
            "finding": f"best switching threshold는 < {hybrid_summary['best_switching_threshold']} interactions이며, strict cold-start가 적은 split에서는 coverage-oriented fallback으로 해석하는 편이 적절합니다.",
        },
        {
            "topic": "content model role",
            "finding": "metadata TF-IDF content model은 정확도는 낮지만 catalog reach를 넓히는 보조 신호로 가치가 있습니다.",
        },
    ]
)
key_findings


,topic,finding
0,default ranker,"weighted_hybrid_alpha_0.85가 전체 Precision@10, N..."
1,weighted hybrid,"best alpha는 0.85이며, CF 단독 대비 accuracy와 coverag..."
2,switching hybrid,"best switching threshold는 < 15 interactions이며,..."
3,content model role,metadata TF-IDF content model은 정확도는 낮지만 catalo...


## 요약
- **기본 운영 모델**: Weighted Hybrid
- **보조 fallback 전략**: sparse-profile user 대상 Switching Hybrid
- **콘텐츠 기반 모델의 역할**: 단독 ranker보다는 cold item 노출과 catalog exploration 보조 신호
- **면접/포트폴리오 메시지**: 정확도만 보는 대신 coverage/diversity까지 함께 비교해, "언제 어떤 모델을 써야 하는가"를 설명할 수 있게 됨
